In [13]:
from transformers import pipeline

pipe = pipeline("zero-shot-classification", model="facebook/bart-large-mnli")

Device set to use cpu


In [14]:
sequence_to_classify = "one day I will see the world"
candidate_labels = ['travel', 'cooking', 'dancing']
pipe(sequence_to_classify, candidate_labels)

{'sequence': 'one day I will see the world',
 'labels': ['travel', 'dancing', 'cooking'],
 'scores': [0.9938650727272034, 0.003273805370554328, 0.002861038316041231]}

In [15]:
CANDIDATE_LABELS = [
    "product news or rumor",
    "advertisement or deal",
    "complaint or criticism",
    "personal anecdote",
    "photography or creative content",
    "humor or shitpost",
    "competitor comparison",
    "technical bug or support",
    "neutral or informational",
    "unrelated to apple iphone"
]

In [16]:
import pandas as pd

df = pd.read_csv(r"/home/ed0uard/Documents/dev/proj-terraform/code/load_data/data.csv")
df

,post_id,author_handle,text
0,at://did:plc:eamjebhx3dz4rrilhidajsf6/app.bsky...,n3rdopolis.bsky.social,...And you'll smile knowing they belched 10 00...
1,at://did:plc:nt6hhsj2ip7bir5srdw4qok7/app.bsky...,apfeltalk.de,NaN
2,at://did:plc:d2y37fe6o6dbvbcysukvkunj/app.bsky...,charleynobody.bsky.social,This is what I love about my 2005 Lexus RX 330...
3,at://did:plc:m2hze6zxa744iberzknpkc3i/app.bsky...,fiftyshadesofwhey.bsky.social,I saw The Odyssey the only way it was meant to...
4,at://did:plc:dqovyc5crdk3blgmrmbl6qe7/app.bsky...,timeline-cleanse.bsky.social,By: Zach Lezniewicz https://www.ashestobeauty...
...,...,...,...
90,at://did:plc:ritv6l6udmzeuwbit4omuq4x/app.bsky...,macrumors.bsky.social,Hands-On With Samsung's Galaxy Z Fold8 the Cl...
91,at://did:plc:anpfslmr5xhyw7oqzai3dprd/app.bsky...,leparisien.fr,Format prix… Découvrez le nouveau smartphone ...
92,at://did:plc:kboucnzkxzmqmatvhes4xlt4/app.bsky...,cait518.bsky.social,复古相机与美颜编辑 PhotoCream Pro 提供超过100种真实胶片滤镜，让您在iP...
93,at://did:plc:darasevwpslhqq6mcy5mey56/app.bsky...,smartdeals.bsky.social,Great Deal! Baseus 15W Fast Charging Car Charg...


In [17]:
from tqdm import tqdm
import pandas as pd

tqdm.pandas()

In [ ]:
def categorize_text(text):
    text = str(text)
    if text is None or text.strip() == "" or text.strip().lower() == "nan":
        return "unknown"
    result = pipe(text, CANDIDATE_LABELS)
    return result["labels"][0]  # Return the label with the highest score


100%|██████████| 95/95 [04:24<00:00,  2.78s/it]


,post_id,author_handle,text,category
0,at://did:plc:eamjebhx3dz4rrilhidajsf6/app.bsky...,n3rdopolis.bsky.social,...And you'll smile knowing they belched 10 00...,unrelated to apple iphone
1,at://did:plc:nt6hhsj2ip7bir5srdw4qok7/app.bsky...,apfeltalk.de,NaN,unknown
2,at://did:plc:d2y37fe6o6dbvbcysukvkunj/app.bsky...,charleynobody.bsky.social,This is what I love about my 2005 Lexus RX 330...,unrelated to apple iphone
3,at://did:plc:m2hze6zxa744iberzknpkc3i/app.bsky...,fiftyshadesofwhey.bsky.social,I saw The Odyssey the only way it was meant to...,humor or shitpost
4,at://did:plc:dqovyc5crdk3blgmrmbl6qe7/app.bsky...,timeline-cleanse.bsky.social,By: Zach Lezniewicz https://www.ashestobeauty...,photography or creative content
...,...,...,...,...
90,at://did:plc:ritv6l6udmzeuwbit4omuq4x/app.bsky...,macrumors.bsky.social,Hands-On With Samsung's Galaxy Z Fold8 the Cl...,competitor comparison
91,at://did:plc:anpfslmr5xhyw7oqzai3dprd/app.bsky...,leparisien.fr,Format prix… Découvrez le nouveau smartphone ...,complaint or criticism
92,at://did:plc:kboucnzkxzmqmatvhes4xlt4/app.bsky...,cait518.bsky.social,复古相机与美颜编辑 PhotoCream Pro 提供超过100种真实胶片滤镜，让您在iP...,photography or creative content
93,at://did:plc:darasevwpslhqq6mcy5mey56/app.bsky...,smartdeals.bsky.social,Great Deal! Baseus 15W Fast Charging Car Charg...,advertisement or deal


In [39]:
from openai import OpenAI
import re

client = OpenAI(
    base_url="https://litellm.skillissue.ovh/v1",
    api_key="sk-REDACTED"
)

import re

def classify_with_openai(text):
    labels_prompt = "\n".join(f"{i}: {label}" for i, label in enumerate(CANDIDATE_LABELS))

    response = client.chat.completions.create(
        model="groq/openai/gpt-oss-20b",
        messages=[
            {"role": "system", "content": "You are a helpful assistant that classifies text into categories. Only return the category number, nothing else. No markdown, no fences, no explanation, just the digit."},
            {"role": "user", "content": f"Categories:\n{labels_prompt}\n\nClassify this text into one of the categories above. Text: {text}"}
        ]
    )
    raw = response.choices[0].message.content.strip()

    match = re.search(r"\d+", raw)
    if not match:
        return "unknown"

    idx = int(match.group())
    if 0 <= idx < len(CANDIDATE_LABELS):
        return CANDIDATE_LABELS[idx]
    return "unknown"

In [40]:

df["category"] = df["text"].progress_apply(classify_with_openai)

df

100%|██████████| 95/95 [00:46<00:00,  2.06it/s]


,post_id,author_handle,text,category
0,at://did:plc:eamjebhx3dz4rrilhidajsf6/app.bsky...,n3rdopolis.bsky.social,...And you'll smile knowing they belched 10 00...,complaint or criticism
1,at://did:plc:nt6hhsj2ip7bir5srdw4qok7/app.bsky...,apfeltalk.de,NaN,unrelated to apple iphone
2,at://did:plc:d2y37fe6o6dbvbcysukvkunj/app.bsky...,charleynobody.bsky.social,This is what I love about my 2005 Lexus RX 330...,personal anecdote
3,at://did:plc:m2hze6zxa744iberzknpkc3i/app.bsky...,fiftyshadesofwhey.bsky.social,I saw The Odyssey the only way it was meant to...,personal anecdote
4,at://did:plc:dqovyc5crdk3blgmrmbl6qe7/app.bsky...,timeline-cleanse.bsky.social,By: Zach Lezniewicz https://www.ashestobeauty...,photography or creative content
...,...,...,...,...
90,at://did:plc:ritv6l6udmzeuwbit4omuq4x/app.bsky...,macrumors.bsky.social,Hands-On With Samsung's Galaxy Z Fold8 the Cl...,competitor comparison
91,at://did:plc:anpfslmr5xhyw7oqzai3dprd/app.bsky...,leparisien.fr,Format prix… Découvrez le nouveau smartphone ...,advertisement or deal
92,at://did:plc:kboucnzkxzmqmatvhes4xlt4/app.bsky...,cait518.bsky.social,复古相机与美颜编辑 PhotoCream Pro 提供超过100种真实胶片滤镜，让您在iP...,advertisement or deal
93,at://did:plc:darasevwpslhqq6mcy5mey56/app.bsky...,smartdeals.bsky.social,Great Deal! Baseus 15W Fast Charging Car Charg...,advertisement or deal


In [41]:
df.to_csv(r"/home/ed0uard/Documents/dev/proj-terraform/code/load_data/data_with_categories.csv", index=False)